In [1]:
!pip install holidays


In [4]:
import pandas as pd
import numpy as np
import holidays

# 1. Load Data and Initial Filtering
# ---------------------------------------------------------
data_path = '../data/agmarknet.csv'

df = pd.read_csv(data_path)
print(df.head())

         State District    Market Commodity Variety Grade Arrival_Date  \
0  Maharashtra     Pune  Baramati     Onion     Red   FAQ   10-07-2024   
1  Maharashtra     Pune  Baramati     Onion     Red   FAQ   20-07-2024   
2  Maharashtra     Pune  Baramati     Onion     Red   FAQ   29-07-2024   
3  Maharashtra     Pune  Baramati     Onion     Red   FAQ   17-08-2024   
4  Maharashtra     Pune  Baramati     Onion     Red   FAQ   03-06-2024   

   Min_Price  Max_Price  Modal_Price  Commodity_Code  
0       1000       3210         2500              23  
1        500       3000         2300              23  
2        900       3000         2200              23  
3       1000       3610         2600              23  
4        300       2300         1550              23  


In [5]:
# 1. Clean up basic string inconsistencies first
df['Market'] = df['Market'].str.strip().str.title()
df['District'] = df['District'].str.strip().str.title()

# 2. How to CHECK for variants (Run this interactively to inspect)
unique_mandis = df['Market'].unique()
print(unique_mandis)

['Baramati' 'Indapur' 'Indapur Apmc' 'Junnar' 'Junnar Apmc'
 'Junnar(Alephata)' 'Junnar(Alephata) Apmc' 'Junnar(Narayangaon)'
 'Junnar(Narayangaon) Apmc' 'Junnar(Otur)' 'Junnar(Otur) Apmc'
 'Khed(Chakan)' 'Khed(Chakan) Apmc' 'Manchar' 'Manchar Apmc' 'Nira' 'Pune'
 'Pune Apmc' 'Pune(Khadiki)' 'Pune(Khadiki) Apmc' 'Pune(Manjri)'
 'Pune(Manjri) Apmc' 'Pune(Moshi)' 'Pune(Moshi) Apmc' 'Pune(Pimpri)'
 'Pune(Pimpri) Apmc' 'Shirur']


In [6]:
df.rename(columns={'State':'state', 'District':'district', 'Market':'mandi_name', 'Commodity':'commodity', 'Variety':'variety','Grade':'grade', 'Arrival_Date':'arrival_date', 'Min_Price':'min_price', 'Max_Price':'max_price', 'Modal_Price':'modal_price', 'Commodity_Code':'commodity_code'}, inplace=True)
print(df.head());

         state district mandi_name commodity variety grade arrival_date  \
0  Maharashtra     Pune   Baramati     Onion     Red   FAQ   10-07-2024   
1  Maharashtra     Pune   Baramati     Onion     Red   FAQ   20-07-2024   
2  Maharashtra     Pune   Baramati     Onion     Red   FAQ   29-07-2024   
3  Maharashtra     Pune   Baramati     Onion     Red   FAQ   17-08-2024   
4  Maharashtra     Pune   Baramati     Onion     Red   FAQ   03-06-2024   

   min_price  max_price  modal_price  commodity_code  
0       1000       3210         2500              23  
1        500       3000         2300              23  
2        900       3000         2200              23  
3       1000       3610         2600              23  
4        300       2300         1550              23  


In [7]:
df.drop(columns='commodity_code', inplace=True)

In [8]:
from thefuzz import process

print("Potential variants to inspect:")
for mandi in unique_mandis:
    # Find the top 3 closest matches for each mandi name
    matches = process.extract(mandi, unique_mandis, limit=3)
    # Filter out exact matches and keep ones with high similarity (e.g., > 85%)
    similar_matches = [m for m in matches if m[1] >= 85 and m[0] != mandi]
    
    if similar_matches:
        print(f"Original: {mandi} -> Similar to: {similar_matches}")

Potential variants to inspect:
Original: Indapur -> Similar to: [('Indapur Apmc', 90)]
Original: Indapur Apmc -> Similar to: [('Indapur', 90), ('Junnar(Alephata) Apmc', 86)]
Original: Junnar -> Similar to: [('Junnar Apmc', 90), ('Junnar(Alephata)', 90)]
Original: Junnar Apmc -> Similar to: [('Junnar', 90), ('Junnar(Alephata) Apmc', 86)]
Original: Junnar(Alephata) -> Similar to: [('Junnar(Alephata) Apmc', 95), ('Junnar', 90)]
Original: Junnar(Alephata) Apmc -> Similar to: [('Junnar(Alephata)', 95), ('Junnar', 90)]
Original: Junnar(Narayangaon) -> Similar to: [('Junnar(Narayangaon) Apmc', 95), ('Junnar', 90)]
Original: Junnar(Narayangaon) Apmc -> Similar to: [('Junnar(Narayangaon)', 95), ('Junnar', 90)]
Original: Junnar(Otur) -> Similar to: [('Junnar', 90), ('Junnar(Otur) Apmc', 90)]
Original: Junnar(Otur) Apmc -> Similar to: [('Junnar', 90), ('Junnar(Otur)', 90)]
Original: Khed(Chakan) -> Similar to: [('Khed(Chakan) Apmc', 90)]
Original: Khed(Chakan) Apmc -> Similar to: [('Khed(Chakan)'

In [ ]:
!pip install thefuzz


In [9]:
mandi_corrections = {
    'Baramati' : 'Baramati',
    'Indapur' : 'Indapur',
    'Indapur Apmc' : 'Indapur',
    'Junnar' : 'Junnar',
    'Junnar Apmc' : 'Junnar',
    'Junnar(Alephata)' : 'Junnar(Alephata)',
    'Junnar(Alephata) Apmc' : 'Junnar(Alephata)',
    'Junnar(Narayangaon)' : 'Junnar(Narayangaon)',
    'Junnar(Narayangaon) Apmc' : 'Junnar(Narayangaon)',
    'Junnar(Otur)' : 'Junnar(Otur)',
    'Junnar(Otur) Apmc' : 'Junnar(Otur)',
    'Khed(Chakan)' : 'Khed(Chakan)',
    'Khed(Chakan) Apmc' : 'Khed(Chakan)',
    'Manchar' : 'Manchar',
    'Manchar Apmc' : 'Manchar',
    'Nira': 'Nira',
    'Pune' : 'Pune',
    'Pune Apmc' : 'Pune',
    'Pune(Khadiki)' : 'Pune(Khadiki)',
    'Pune(Khadiki) Apmc' : 'Pune(Khadiki)',
    'Pune(Manjri)' : 'Pune(Manjri)',
    'Pune(Manjri) Apmc' : 'Pune(Manjri)',
    'Pune(Moshi)' : 'Pune(Moshi)',
    'Pune(Moshi) Apmc' : 'Pune(Moshi)',
    'Pune(Pimpri)' : 'Pune(Pimpri)',
    'Pune(Pimpri) Apmc' : 'Pune(Pimpri)',
    'Shirur' : 'Shirur'
}

df['mandi_name'] = df['mandi_name'].replace(mandi_corrections)

In [10]:
unique_mandis = df['mandi_name'].unique()
print(unique_mandis)

['Baramati' 'Indapur' 'Junnar' 'Junnar(Alephata)' 'Junnar(Narayangaon)'
 'Junnar(Otur)' 'Khed(Chakan)' 'Manchar' 'Nira' 'Pune' 'Pune(Khadiki)'
 'Pune(Manjri)' 'Pune(Moshi)' 'Pune(Pimpri)' 'Shirur']


In [11]:
# 1. Clean up basic string inconsistencies first
df['variety'] = df['variety'].str.strip().str.title()

# 2. How to CHECK for variants (Run this interactively to inspect)
unique_varieties = df['variety'].unique()
print(unique_varieties)

['Red' 'Other' 'Local']


In [12]:
# 1. Clean up basic string inconsistencies first
df['grade'] = df['grade'].str.strip().str.title()

# 2. How to CHECK for variants (Run this interactively to inspect)
unique_grades = df['grade'].unique()
print(unique_grades)

['Faq' 'Local']


In [13]:
# Convert date to datetime object
df['arrival_date'] = pd.to_datetime(df['arrival_date'], format='%d-%m-%Y')

In [14]:
# Filter for Onion commodity
df = df[df['commodity'].str.lower() == 'onion'].copy()
df = df.drop(columns=['commodity_code', 'grade'], errors='ignore')

In [15]:
print(df.head())

         state district mandi_name commodity variety arrival_date  min_price  \
0  Maharashtra     Pune   Baramati     Onion     Red   2024-07-10       1000   
1  Maharashtra     Pune   Baramati     Onion     Red   2024-07-20        500   
2  Maharashtra     Pune   Baramati     Onion     Red   2024-07-29        900   
3  Maharashtra     Pune   Baramati     Onion     Red   2024-08-17       1000   
4  Maharashtra     Pune   Baramati     Onion     Red   2024-06-03        300   

   max_price  modal_price  
0       3210         2500  
1       3000         2300  
2       3000         2200  
3       3610         2600  
4       2300         1550  


In [16]:
# Clean up string inconsistencies
df['mandi_name'] = df['mandi_name'].str.strip().str.title()
df['district'] = df['district'].str.strip().str.title()
df['variety'] = df['variety'].str.strip().str.title()
#df['grade'] = df['grade'].str.strip().str.title()

In [17]:
# Sort data to ensure chronological order before any operations
df = df.sort_values(by=['mandi_name', 'variety', 'arrival_date'])

In [18]:
print(df.head(10))

           state district mandi_name commodity variety arrival_date  \
179  Maharashtra     Pune   Baramati     Onion   Other   2023-05-15   
81   Maharashtra     Pune   Baramati     Onion   Other   2023-05-18   
109  Maharashtra     Pune   Baramati     Onion     Red   2023-05-17   
25   Maharashtra     Pune   Baramati     Onion     Red   2023-05-20   
82   Maharashtra     Pune   Baramati     Onion     Red   2023-05-22   
83   Maharashtra     Pune   Baramati     Onion     Red   2023-05-24   
233  Maharashtra     Pune   Baramati     Onion     Red   2023-05-27   
110  Maharashtra     Pune   Baramati     Onion     Red   2023-05-29   
26   Maharashtra     Pune   Baramati     Onion     Red   2023-05-31   
111  Maharashtra     Pune   Baramati     Onion     Red   2023-06-03   

     min_price  max_price  modal_price  
179        200        750          600  
81         200        750          600  
109        200        700          550  
25         200        801          600  
82         20

In [19]:
print("Analyzing date ranges and filtering sparse data...")

# Calculate the actual start date, end date, and total records for each combination
group_stats = df.groupby(['mandi_name', 'variety']).agg(
    start_date=('arrival_date', 'min'),
    end_date=('arrival_date', 'max'),
    record_count=('arrival_date', 'count')
).reset_index()

# Calculate the total possible days in each group's specific active window
group_stats['possible_days'] = (group_stats['end_date'] - group_stats['start_date']).dt.days + 1

# Calculate data density (what percentage of days actually have a price recorded?)
group_stats['density'] = group_stats['record_count'] / group_stats['possible_days']

print("\n--- Mandi Diagnostics ---")
print(group_stats[['mandi_name', 'variety', 'start_date', 'record_count', 'density']].head(20))

Analyzing date ranges and filtering sparse data...

--- Mandi Diagnostics ---
             mandi_name variety start_date  record_count   density
0              Baramati   Other 2023-05-15             2  0.500000
1              Baramati     Red 2023-05-17           250  0.375375
2               Indapur   Other 2021-05-05            20  0.011758
3               Indapur     Red 2021-04-09           129  0.076877
4                Junnar   Local 2022-04-23             6  0.015424
5                Junnar   Other 2021-02-25           394  0.217200
6      Junnar(Alephata)   Other 2021-02-23           645  0.353231
7   Junnar(Narayangaon)   Other 2022-06-23           839  0.626587
8          Junnar(Otur)   Local 2025-03-07             1  1.000000
9          Junnar(Otur)   Other 2021-02-25           368  0.201754
10         Junnar(Otur)     Red 2025-01-16            22  0.056555
11         Khed(Chakan)   Other 2021-02-24          1223  0.670504
12              Manchar   Other 2021-02-22         

In [20]:
# Filter Rule: Must have at least 300 total records AND at least 30% density in its active window
valid_groups = group_stats[(group_stats['record_count'] >= 300) & (group_stats['density'] >= 0.30)]

print(f"\nKeeping {len(valid_groups)} dense combinations out of {len(group_stats)}.")


Keeping 9 dense combinations out of 23.


In [21]:
# =========================================================
# STEP 3: Outlier Detection & Removal (IQR Method)
# =========================================================
print("Removing price outliers...")
def remove_price_outliers(group, col='modal_price'):
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Replace outliers with NaN to be forward-filled later
    group.loc[(group[col] < lower_bound) | (group[col] > upper_bound), col] = np.nan
    return group

df = df.groupby(['mandi_name', 'variety']).apply(remove_price_outliers).reset_index(drop=True)


Removing price outliers...


C:\Users\smita\AppData\Local\Temp\ipykernel_144020\4144140127.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['mandi_name', 'variety']).apply(remove_price_outliers).reset_index(drop=True)


In [22]:
# =========================================================
# STEP 3.5: Resolve Same-Day Duplicates
# =========================================================
print("Resolving multiple price entries for the same day...")

# Define how to handle the columns when squashing duplicate days together
aggregation_dict = {
    'min_price': 'mean',   # Take the average of the prices
    'max_price': 'mean',
    'modal_price': 'mean'
}

# If you kept the 'commodity_name' column, we just keep the first instance of the word
if 'commodity' in df.columns:
    aggregation_dict['commodity'] = 'first'

# Group by everything that identifies a unique timeline, PLUS the specific date
df = df.groupby(
    ['mandi_name', 'district', 'state','variety', 'arrival_date']
).agg(aggregation_dict).reset_index()

print("Duplicates resolved. Ready for reindexing.")

# =========================================================
# STEP 4: Dynamic Reindexing & Zero-Arrival Flagging
# =========================================================
print("Dynamically reindexing and flagging zero-arrival days...")

# 1. THE TRICK: Add a flag to all our real, existing raw data
df['is_real_trade'] = 1

def fill_missing_dates_dynamic(group):
    actual_start = group['arrival_date'].min()
    actual_end = group['arrival_date'].max()
    
    group = group.set_index('arrival_date')
    
    # Create the daily range spanning this group's active period
    full_date_range = pd.date_range(start=actual_start, end=actual_end, freq='D')
    
    # Reindex to insert missing days. 
    # For all the brand new days created here, 'is_real_trade' will become NaN!
    group = group.reindex(full_date_range)
    group.index.name = 'arrival_date'
    
    # 2. THE FLAG: Fill the NaNs with 0 to explicitly flag the zero-arrival days
    group['is_real_trade'] = group['is_real_trade'].fillna(0).astype(int)
    
    static_cols = ['mandi_name', 'district', 'state', 'variety']
    group[static_cols] = group[static_cols].ffill().bfill()
    
    price_cols = ['min_price', 'max_price', 'modal_price']
    group[price_cols] = group[price_cols].ffill()
    
    return group.reset_index()

# Apply the updated function
df_continuous = df.groupby(['mandi_name', 'district', 'state', 'variety']).apply(fill_missing_dates_dynamic).reset_index(drop=True)

Resolving multiple price entries for the same day...
Duplicates resolved. Ready for reindexing.
Dynamically reindexing and flagging zero-arrival days...


C:\Users\smita\AppData\Local\Temp\ipykernel_144020\3670777500.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_continuous = df.groupby(['mandi_name', 'district', 'state', 'variety']).apply(fill_missing_dates_dynamic).reset_index(drop=True)


In [23]:
print(df.columns)

Index(['mandi_name', 'district', 'state', 'variety', 'arrival_date',
       'min_price', 'max_price', 'modal_price', 'commodity', 'is_real_trade'],
      dtype='object')


In [24]:
duplicates = df[df.duplicated(subset=['arrival_date', 'mandi_name', 'variety'], keep=False)]
print(duplicates.sort_values(by=['mandi_name', 'arrival_date']).head(10))

Empty DataFrame
Columns: [mandi_name, district, state, variety, arrival_date, min_price, max_price, modal_price, commodity, is_real_trade]
Index: []


In [25]:
# =========================================================
# STEP 5: Calendar Features & Holiday Flags
# =========================================================
print("Generating calendar and holiday features...")
ind_holidays = holidays.India(years=range(2021, 2027))

df_continuous['day_of_week'] = df_continuous['arrival_date'].dt.dayofweek
df_continuous['month'] = df_continuous['arrival_date'].dt.month
df_continuous['day_of_year'] = df_continuous['arrival_date'].dt.dayofyear
df_continuous['is_weekend'] = df_continuous['day_of_week'].isin([5, 6]).astype(int)
df_continuous['is_holiday'] = df_continuous['arrival_date'].apply(lambda x: 1 if x in ind_holidays else 0)

Generating calendar and holiday features...


In [27]:
# =========================================================
# STEP: 7-Day Horizon Feature Engineering
# =========================================================
print("Calculating 7-Day Horizon specific lags and rolling features...")

# 1. Base Lags (The most recent known price is 7 days ago)
df_continuous['price_lag_7'] = df_continuous.groupby('mandi_name')['modal_price'].shift(7)
df_continuous['price_lag_8'] = df_continuous.groupby('mandi_name')['modal_price'].shift(8)
df_continuous['price_lag_9'] = df_continuous.groupby('mandi_name')['modal_price'].shift(9)
df_continuous['price_lag_14'] = df_continuous.groupby('mandi_name')['modal_price'].shift(14)
df_continuous['price_lag_30'] = df_continuous.groupby('mandi_name')['modal_price'].shift(30)

# 2. Rolling Means (Must be calculated relative to price_lag_7, NOT lag 1!)
# This calculates the 7-day average of the week leading up to 7 days ago
df_continuous['price_roll_mean_7'] = df_continuous.groupby('mandi_name')['price_lag_7'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
df_continuous['price_roll_std_7'] = df_continuous.groupby('mandi_name')['price_lag_7'].transform(lambda x: x.rolling(window=7, min_periods=1).std())

df_continuous['price_roll_mean_30'] = df_continuous.groupby('mandi_name')['price_lag_7'].transform(lambda x: x.rolling(window=30, min_periods=1).mean())

# 3. Expanding Mean (Up to 7 days ago)
df_continuous['price_expanding_mean'] = df_continuous.groupby('mandi_name')['price_lag_7'].transform(lambda x: x.expanding().mean())

# 4. Drop Rows with NaNs caused by the 30-day shift
# Since we shifted by up to 30 days, the first 30 days of every mandi will be blank.
df_final_7day = df_continuous.dropna(subset=['modal_price', 'price_roll_mean_30', 'price_lag_30']).copy()

# Save the 7-day specific dataset
# csv_filename = 'prepared_onion_pune_7day_master.csv'
# df_final_7day.to_csv(csv_filename, index=False)

# print(f"Data pipeline complete! Saved {len(df_final_7day)} records for 7-Day training.")

Calculating 7-Day Horizon specific lags and rolling features...


In [28]:
# =========================================================
# STEP 7: Fourier Terms
# =========================================================
df_final_7day['sin_365_1'] = np.sin(2 * np.pi * df_final_7day['day_of_year'] / 365.25)
df_final_7day['cos_365_1'] = np.cos(2 * np.pi * df_final_7day['day_of_year'] / 365.25)
df_final_7day['sin_365_2'] = np.sin(4 * np.pi * df_final_7day['day_of_year'] / 365.25)
df_final_7day['cos_365_2'] = np.cos(4 * np.pi * df_final_7day['day_of_year'] / 365.25)

In [30]:
# =========================================================
# STEP 8: Target Variable Creation
# =========================================================
# Tomorrow's price
df_final_7day['target_price'] = df_continuous.groupby('mandi_name')['modal_price'].shift(-7)

In [32]:
# =========================================================
# STEP 9: Final Cleanup & Save
# =========================================================
print("Cleaning up edge cases and saving master dataset...")
df_final_7day = df_final_7day.dropna(subset=['target_price', 'price_roll_mean_30', 'price_lag_7']).copy()

csv_filename = 'prepared_onion_pune_dynamic_master_7day.csv'
df_final_7day.to_csv(csv_filename, index=False)

print(f"Data pipeline complete! Saved {len(df_final_7day)} clean records to '{csv_filename}'.")

Cleaning up edge cases and saving master dataset...
Data pipeline complete! Saved 24811 clean records to 'prepared_onion_pune_dynamic_master_7day.csv'.
